# QuantJourney SDK - Factor Risk Model Construction

This notebook demonstrates a QuantJourney SDK workflow that builds sample, shrinkage and factor-model covariance estimates from price and factor returns.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


## Factor and Regime Helpers

In [ ]:
def rolling_betas(y: pd.Series, x: pd.DataFrame, window: int=126) -> pd.DataFrame:
    data = pd.concat([y.rename('asset'), x], axis=1).dropna()
    rows = []
    for i in range(window, len(data)):
        chunk = data.iloc[i - window:i]
        yy = chunk['asset'].to_numpy()
        xx = np.column_stack([np.ones(len(chunk)), chunk[x.columns].to_numpy()])
        beta = np.linalg.lstsq(xx, yy, rcond=None)[0][1:]
        rows.append(dict(date=data.index[i], **{col: beta[j] for j, col in enumerate(x.columns)}))
    return pd.DataFrame(rows).set_index('date') if rows else pd.DataFrame(columns=x.columns)

def zscore(s: pd.Series, window: int=252) -> pd.Series:
    return (s - s.rolling(window).mean()) / s.rolling(window).std()


In [ ]:
assets = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META']
factor_symbols = ['SPY', 'QQQ', 'IWM', 'TLT', 'GLD']
ff_factors = qj.ff.get_factors(region='US')
prices, volumes = price_panel(assets + factor_symbols)
ret = returns(prices).dropna()
asset_ret = ret[assets]
factor_ret = ret[factor_symbols]


In [ ]:
sample_cov = asset_ret.cov() * 252
diagonal = pd.DataFrame(np.diag(np.diag(sample_cov)), index=assets, columns=assets)
shrinkage_cov = 0.75 * sample_cov + 0.25 * diagonal
betas = pd.DataFrame({asset: rolling_betas(asset_ret[asset], factor_ret, window=252).iloc[-1] for asset in assets}).T
factor_cov = factor_ret.cov() * 252
systematic = betas @ factor_cov @ betas.T
resid = asset_ret - factor_ret @ betas.T
specific = pd.DataFrame(np.diag(resid.var() * 252), index=assets, columns=assets)
factor_model_cov = systematic + specific
display(betas)
display(pd.DataFrame({'sample_vol': np.sqrt(np.diag(sample_cov)), 'shrinkage_vol': np.sqrt(np.diag(shrinkage_cov)), 'factor_model_vol': np.sqrt(np.diag(factor_model_cov))}, index=assets))
plt.imshow(factor_model_cov.corr(), cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(assets)), assets, rotation=45)
plt.yticks(range(len(assets)), assets)
plt.title('Factor-model covariance correlation')
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.